# Semana 5 — Sistemas Multi-Qubit y Entrelazamiento
## Cuando los qubits no pueden describirse por separado

**Objetivo:** Construir estados de múltiples qubits con producto tensorial, identificar estados entrelazados, crear los estados de Bell y entender la no-localidad cuántica.

**Hito verificable:** Puedes crear los 4 estados de Bell, verificar su entrelazamiento y simular la correlación cuántica que viola las desigualdades de Bell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('Librerías importadas correctamente')

## Ejercicio 5.1 — Producto tensorial y espacio de dos qubits
El espacio de 2 qubits tiene dimensión 4. La base es {|00⟩, |01⟩, |10⟩, |11⟩}.

In [ ]:
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Los 4 estados base de 2 qubits
ket_00 = np.kron(ket_0, ket_0)
ket_01 = np.kron(ket_0, ket_1)
ket_10 = np.kron(ket_1, ket_0)
ket_11 = np.kron(ket_1, ket_1)

print('Base computacional de 2 qubits:')
for nombre, v in [('|00⟩', ket_00), ('|01⟩', ket_01), ('|10⟩', ket_10), ('|11⟩', ket_11)]:
    print(f'  {nombre} = {v}')

print()
print('Producto tensorial de operadores:')
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
X = np.array([[0, 1], [1, 0]], dtype=complex)

HH = np.kron(H, H)   # H aplicado a ambos qubits
HI = np.kron(H, np.eye(2, dtype=complex))   # H sólo al primer qubit

print(f'  H⊗H |00⟩ = {np.round(HH @ ket_00, 4)}')
print(f'  H⊗I |00⟩ = {np.round(HI @ ket_00, 4)}')

## Ejercicio 5.2 — La puerta CNOT
La puerta más importante de 2 qubits. Flipa el qubit objetivo si el control es |1⟩.

In [ ]:
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

print('Tabla de verdad de CNOT (control=qubit 0, target=qubit 1):')
estados_base = [('|00⟩', ket_00), ('|01⟩', ket_01), ('|10⟩', ket_10), ('|11⟩', ket_11)]
nombres_base = {str(v.tolist()): k for k, v in estados_base}

for nombre, v in estados_base:
    resultado = CNOT @ v
    idx = np.argmax(np.abs(resultado))
    nombres_salida = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
    print(f'  CNOT {nombre} = {nombres_salida[idx]}')

print()
print(f'CNOT es unitaria: {np.allclose(CNOT.conj().T @ CNOT, np.eye(4))}')
print(f'CNOT es su propio inverso: {np.allclose(CNOT @ CNOT, np.eye(4))}')

## Ejercicio 5.3 — Los estados de Bell: máximo entrelazamiento
Los 4 estados de Bell son los estados maximalmente entrelazados de 2 qubits.

In [ ]:
# Crear estado de Bell |Φ+⟩ = (|00⟩ + |11⟩)/√2
# Circuito: H en qubit 0, luego CNOT
psi_inicial = ket_00.copy()
psi_despues_H = HI @ psi_inicial  # H en primer qubit
bell_phi_plus = CNOT @ psi_despues_H

print('Construcción de |Φ+⟩ = (|00⟩ + |11⟩)/√2:')
print(f'  |00⟩ → H⊗I → {np.round(psi_despues_H, 4)}')
print(f'  → CNOT → {np.round(bell_phi_plus, 4)}')
print()

# Los 4 estados de Bell
def crear_bell(x, y):
    """Crea estado de Bell a partir de |xy⟩."""
    entrada = np.kron(np.array([1,0] if x==0 else [0,1], dtype=complex),
                      np.array([1,0] if y==0 else [0,1], dtype=complex))
    return CNOT @ HI @ entrada

bells = {
    '|Φ+⟩': crear_bell(0, 0),
    '|Φ−⟩': crear_bell(0, 1),  # Nota: Z en qubit 1 del |Φ+⟩
    '|Ψ+⟩': crear_bell(1, 0),
    '|Ψ−⟩': crear_bell(1, 1),
}

print('Los 4 estados de Bell:')
for nombre, b in bells.items():
    probs = np.round(np.abs(b)**2, 4)
    print(f'  {nombre} = {np.round(b, 4)}')
    print(f'       probs = {probs}')

# Verificar que forman una base ortonormal
bell_list = list(bells.values())
print()
print('Verificar ortogonalidad (todos los productos cruzados deben ser ≈0):')
for i, (ni, bi) in enumerate(bells.items()):
    for j, (nj, bj) in enumerate(bells.items()):
        if i < j:
            prod = np.dot(bi.conj(), bj)
            print(f'  ⟨{ni}|{nj}⟩ = {np.round(prod, 10)}')

## Ejercicio 5.4 — Detectar entrelazamiento: rango de Schmidt
Un estado de 2 qubits es producto separable si y solo si su rango de Schmidt es 1.

In [ ]:
def rango_schmidt(psi, dim_A=2, dim_B=2):
    """Calcula el rango de Schmidt de un estado bipartito."""
    # Reshaping el vector de estado en una matriz dim_A × dim_B
    M = psi.reshape(dim_A, dim_B)
    # SVD: los valores singulares dan la descomposición de Schmidt
    singular_values = np.linalg.svd(M, compute_uv=False)
    # El rango de Schmidt es el número de valores singulares no nulos
    rango = np.sum(np.abs(singular_values) > 1e-10)
    return int(rango), singular_values

estados_test = [
    ('|00⟩ (separable)',          ket_00),
    ('|+⟩⊗|+⟩ (separable)',       np.kron((ket_0+ket_1)/np.sqrt(2), (ket_0+ket_1)/np.sqrt(2))),
    ('|Φ+⟩ (entrelazado)',         bells['|Φ+⟩']),
    ('|Ψ−⟩ (entrelazado)',         bells['|Ψ−⟩']),
    ('(|00⟩+|01⟩+|10⟩)/√3 (parcial)', (ket_00+ket_01+ket_10)/np.sqrt(3)),
]

print(f'{"Estado":40}  {"Schmidt":8}  {"Vals singulares"}')
print('-' * 75)
for nombre, psi in estados_test:
    r, sv = rango_schmidt(psi)
    tipo = '✅ separable' if r == 1 else '🔗 entrelazado'
    print(f'{nombre:40}  {tipo:16}  {np.round(sv[sv > 1e-10], 4)}')

## Ejercicio 5.5 — Correlaciones cuánticas vs. clásicas
Al medir uno de los qubits del estado de Bell, el otro queda determinado instantáneamente.

In [ ]:
def medir_primer_qubit(psi_2q, n_shots=10000):
    """Mide el primer qubit de un estado de 2 qubits y colapsa al segundo."""
    # P(medir 0 en qubit A) = |ψ_00|² + |ψ_01|²
    p_A0 = abs(psi_2q[0])**2 + abs(psi_2q[1])**2
    resultados_A = np.random.choice([0, 1], size=n_shots, p=[p_A0, 1-p_A0])
    
    # Estado del qubit B condicionado a A
    # Si A=0: |ψ_B|A=0⟩ ∝ ψ_00|0⟩ + ψ_01|1⟩
    psi_B_dado_A0 = psi_2q[:2] / np.linalg.norm(psi_2q[:2])
    psi_B_dado_A1 = psi_2q[2:] / np.linalg.norm(psi_2q[2:])
    
    resultados_B = []
    for rA in resultados_A:
        if rA == 0:
            p_B0 = abs(psi_B_dado_A0[0])**2
        else:
            p_B0 = abs(psi_B_dado_A1[0])**2
        resultados_B.append(np.random.choice([0, 1], p=[p_B0, 1-p_B0]))
    
    return np.array(resultados_A), np.array(resultados_B)

print('Correlaciones en el estado de Bell |Φ+⟩ = (|00⟩+|11⟩)/√2')
print('Al medir A, el resultado de B queda determinado.')
print()

rA, rB = medir_primer_qubit(bells['|Φ+⟩'], n_shots=10000)

print(f'  P(A=0): {np.mean(rA==0):.4f}  P(A=1): {np.mean(rA==1):.4f}')
print(f'  P(B=0|A=0): {np.mean(rB[rA==0]==0):.4f}  (debe ser ≈1.0)')
print(f'  P(B=1|A=1): {np.mean(rB[rA==1]==1):.4f}  (debe ser ≈1.0)')
correlacion = np.corrcoef(rA, rB)[0,1]
print(f'  Correlación A-B: {correlacion:.4f}  (debe ser ≈+1.0 para |Φ+⟩)')
print()
rA2, rB2 = medir_primer_qubit(bells['|Ψ−⟩'], n_shots=10000)
correlacion2 = np.corrcoef(rA2, rB2)[0,1]
print(f'Para |Ψ−⟩ correlación A-B: {correlacion2:.4f}  (debe ser ≈-1.0)')

## Mini reto — Completar antes de avanzar a la Semana 6

Implementa el circuito de teleportación cuántica:
1. Alice tiene un qubit desconocido |ψ⟩ = α|0⟩ + β|1⟩
2. Alice y Bob comparten un par de Bell |Φ+⟩
3. Alice mide sus 2 qubits y envía 2 bits clásicos a Bob
4. Bob aplica correcciones y recupera |ψ⟩

Verifica que el estado en Bob tras las correcciones es idéntico al estado original de Alice.

In [ ]:
# TU CÓDIGO AQUÍ
def teleportacion(psi_alice: np.ndarray) -> np.ndarray:
    """Simula la teleportación cuántica. Devuelve el estado en Bob."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 5

Antes de avanzar, comprueba que puedes:

- [ ] Construir estados de 2 qubits con np.kron
- [ ] Aplicar CNOT y verificar su tabla de verdad
- [ ] Crear los 4 estados de Bell y verificar su ortogonalidad
- [ ] Detectar entrelazamiento con el rango de Schmidt
- [ ] Simular correlaciones en estados entrelazados

## Reflexión (escribe aquí tu respuesta)

**¿Por qué los estados entrelazados no pueden describirse como producto de estados individuales?**

*(tu respuesta aquí)*

**¿La teleportación cuántica viola la relatividad especial? ¿Por qué?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 6 — Circuitos Cuánticos con Qiskit*